In [1]:
using BenchmarkTools

In [2]:
findAllFactors(n::Integer) = filter(k -> n % k == 0, 1:n)

findAllFactors (generic function with 1 method)

In [3]:
@btime findAllFactors(2^4*5^3*3^7)

  2.322 ms (5 allocations: 33.38 MiB)


160-element Vector{Int64}:
       1
       2
       3
       4
       5
       6
       8
       9
      10
      12
       ⋮
  437400
  486000
  546750
  729000
  874800
 1093500
 1458000
 2187000
 4374000

In [4]:
isPrime(n::Integer) = length(findAllFactors(n)) == 2

isPrime (generic function with 1 method)

In [5]:
nextPrime(n::Integer) = isPrime(n+1) ? n+1 : nextPrime(n+1)

nextPrime (generic function with 1 method)

In [6]:
nextPrime(16), nextPrime(26), nextPrime(1_000_000)

(17, 29, 1000003)

In [7]:
isPrime(11)

true

In [8]:
getPrimes(n) = filter(isPrime,2:n)

getPrimes (generic function with 1 method)

In [9]:
getPrimes(100)

25-element Vector{Int64}:
  2
  3
  5
  7
 11
 13
 17
 19
 23
 29
  ⋮
 59
 61
 67
 71
 73
 79
 83
 89
 97

In [10]:
function getPrimes2(n::Integer)
  local primes = Int[]
  local k = 2
  while k < n
    push!(primes, k)
    k = nextPrime(k)
  end
  primes
end

getPrimes2 (generic function with 1 method)

In [11]:
getPrimes2(100)

25-element Vector{Int64}:
  2
  3
  5
  7
 11
 13
 17
 19
 23
 29
  ⋮
 59
 61
 67
 71
 73
 79
 83
 89
 97

In [12]:
function isPerfect(n::Integer)
  A=findAllFactors(n)
  pop!(A)
  sum(A)==n
end


isPerfect (generic function with 1 method)

In [13]:
isPerfect2(n::Integer) = sum(findAllFactors(n)) == 2n

isPerfect2 (generic function with 1 method)

In [14]:
@time isPerfect(100_000)

  0.000064 seconds (5 allocations: 784.422 KiB)


false

In [15]:
@time isPerfect2(100_000)

  0.000063 seconds (5 allocations: 784.422 KiB)


false

In [16]:
@btime isPerfect(100_000)

  49.375 μs (5 allocations: 784.42 KiB)


false

In [17]:
@btime isPerfect2(100_000)

  49.333 μs (5 allocations: 784.42 KiB)


false

In [18]:
@benchmark isPerfect(100_000)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  49.250 μs … 175.708 μs  ┊ GC (min … max): 0.00% … 59.45%
 Time  (median):     55.792 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   59.323 μs ±  13.419 μs  ┊ GC (mean ± σ):  5.09% ± 10.85%

  ▁▁  ▇██▄▃▃▂▁▁▁                                         ▂▂▂▂  ▂
  ██▇▇███████████▇▆▄▃▄▁▁▁▁▃▁▄▁▁▃▁▁▃▁▁▁▁▁▁▁▁▁▃▁▁▁▁▄▁▁▄▁▁▃▇█████ █
  49.2 μs       Histogram: log(frequency) by time       115 μs <

 Memory estimate: 784.42 KiB, allocs estimate: 5.

In [19]:
digits(1234)

4-element Vector{Int64}:
 4
 3
 2
 1

In [20]:
function isHappy(n::Integer)
  if n==1
    return true
  elseif n==4
    return false
  else
    local d = digits(n)
    local sum = 0
    for i=1:length(d)
      sum += d[i]^2
    end
    return isHappy(sum)
  end
end

isHappy (generic function with 1 method)

In [21]:
function isHappy2(n::Integer)
  if n==1
    return true
  elseif n==4
    return false
  else
    return isHappy2(sum(x->x^2,digits(n)))
  end
end


isHappy2 (generic function with 1 method)

In [22]:
isHappy3(n::Integer) = n == 1 ? true : n == 4 ? false : isHappy3(sum(x->x^2,digits(n)))

isHappy3 (generic function with 1 method)

In [23]:
filter(isHappy3, 1:100)

20-element Vector{Int64}:
   1
   7
  10
  13
  19
  23
  28
  31
  32
  44
  49
  68
  70
  79
  82
  86
  91
  94
  97
 100

In [24]:
join(filter(isHappy3, 1:100)," ")

"1 7 10 13 19 23 28 31 32 44 49 68 70 79 82 86 91 94 97 100"

In [25]:
@btime isHappy(1_234)
@btime isHappy2(1_234)
@btime isHappy3(1_234)

  181.786 ns (24 allocations: 960 bytes)
  177.858 ns (24 allocations: 960 bytes)
  178.571 ns (24 allocations: 960 bytes)


false

In [26]:
n = big(2)^89-1

618970019642690137449562111

In [27]:
isPrime(n::Integer)= length(findAllFactors(n))==2

isPrime (generic function with 1 method)

In [28]:
@btime isPrime(1_000_003)

  500.375 μs (5 allocations: 7.64 MiB)


true

### Speedup #1

In [29]:
function findAllFactors2(n::Integer)
  factors = [1]
  for i=2:n-1
    if n % i ==0
      push!(factors,i)
    end
  end
  push!(factors,n) # n is always a factor of itself
end

findAllFactors2 (generic function with 1 method)

In [30]:
@btime findAllFactors(49_000_000)

  26.930 ms (5 allocations: 373.85 MiB)


147-element Vector{Int64}:
        1
        2
        4
        5
        7
        8
       10
       14
       16
       20
        ⋮
  3062500
  3500000
  4900000
  6125000
  7000000
  9800000
 12250000
 24500000
 49000000

In [34]:
@btime findAllFactors2(49_000_000)

  23.462 ms (7 allocations: 3.38 KiB)


147-element Vector{Int64}:
        1
        2
        4
        5
        7
        8
       10
       14
       16
       20
        ⋮
  3062500
  3500000
  4900000
  6125000
  7000000
  9800000
 12250000
 24500000
 49000000

In [37]:
isPrime2(n::Integer) = length(findAllFactors2(n))==2

isPrime2 (generic function with 1 method)

In [38]:
@btime isPrime2(1_000_003)

  448.208 μs (3 allocations: 128 bytes)


true

### Speedup #2: reducing the checked factors

In [39]:
function findAllFactors3(n::Integer)
  local factors = [1]
  for i=2:n÷2
    if n % i ==0
      push!(factors,i)
    end
  end
  push!(factors,n)
end

findAllFactors3 (generic function with 1 method)

In [40]:
@btime findAllFactors3(49_000_000)

  11.487 ms (7 allocations: 3.38 KiB)


147-element Vector{Int64}:
        1
        2
        4
        5
        7
        8
       10
       14
       16
       20
        ⋮
  3062500
  3500000
  4900000
  6125000
  7000000
  9800000
 12250000
 24500000
 49000000

In [41]:
isPrime3(n::Integer) = length(findAllFactors3(n)) == 2

isPrime3 (generic function with 1 method)

In [42]:
@btime isPrime3(1_000_003)

  224.458 μs (3 allocations: 128 bytes)


true

#### Speedup #3: Notice that factors come in pairs 

In [50]:
function findAllFactors4(n::Integer)
  local x = round(Int,sqrt(n)) # closest integer to sqrt(n)
  local factors = [1,n]
  local j=2 # keep track where to insert elements
  for k=2:x
    if n%k==0
      # Insert the new factors in the middle of the factors array.
      # If k^2 is n, just add k, otherwise add the pairs.
      splice!(factors,j:(j-1),k^2 == n ? [k] : [k,div(n,k)])
      j+=1
    end
  end
  factors
end

findAllFactors4 (generic function with 1 method)

In [51]:
@btime findAllFactors4(49_000_000)

  4.643 μs (226 allocations: 14.52 KiB)


147-element Vector{Int64}:
        1
        2
        4
        5
        7
        8
       10
       14
       16
       20
        ⋮
  3062500
  3500000
  4900000
  6125000
  7000000
  9800000
 12250000
 24500000
 49000000

In [52]:
isPrime4(n::Integer) = length(findAllFactors4(n))==2

isPrime4 (generic function with 1 method)

In [53]:
@btime isPrime4(1_000_003)

  458.335 ns (2 allocations: 80 bytes)


true

#### Speedup #4: don't use factors at all

In [54]:
function isPrime5(n::Integer)
  for k=2:floor(Int,sqrt(n)) # integer nearest sqrt(n)
    if n%k==0
      return false
    end
  end
  true
end

isPrime5 (generic function with 1 method)

In [55]:
@btime isPrime5(1_000_003)

  447.601 ns (0 allocations: 0 bytes)


true

#### Speedup #5: skip all even numbers

In [56]:
function isPrime6(n::Integer)
  if n == 1
  	return false
  elseif n == 2
    return true
  elseif n%2==0
    return false
  end
  for k=3:2:floor(Int,sqrt(n)) # odd integers to sqrt(n)
    if n%k==0
      return false
    end
  end
  true
end

isPrime6 (generic function with 1 method)

In [57]:
@btime isPrime6(1_000_003)

  223.807 ns (0 allocations: 0 bytes)


true

In [58]:
nextPrime3(n::Integer) = isPrime6(n+1) ? n+1 : nextPrime3(n+1)

nextPrime3 (generic function with 1 method)

In [59]:
function getPrimes3(m::Integer) ## return all primes up to n using
  # the sieve of Eratosthenes
  local is_prime=trues(m) ## assume all are prime
  local k=2
  while k < sqrt(m)
    is_prime[2*k:k:m] .= false # all multiples of k are not prime
    k = nextPrime(k+1) # find the next prime after k
  end
  findall(is_prime)[2:end]
end

getPrimes3 (generic function with 1 method)

In [60]:
@btime getPrimes3(10_000)

  22.958 μs (309 allocations: 77.64 KiB)


1895-element Vector{Int64}:
    2
    3
    5
    7
    9
   11
   13
   17
   19
   23
    ⋮
 9941
 9949
 9957
 9967
 9969
 9973
 9981
 9987
 9993

In [87]:
join(getPrimes(100), " ")

"2 3 5 7 11 13 17 19 23 29 31 37 41 43 47 53 59 61 67 71 73 79 83 89 97"

In [88]:
@btime getPrimes(1_000_000)
# @btime getPrimes2(10_000)
@btime getPrimes3(1_000_000)

  6.143 ms (10 allocations: 1.32 MiB)
  6.065 ms (4013 allocations: 5.49 MiB)


122896-element Vector{Int64}:
      2
      3
      5
      7
      9
     11
     13
     17
     19
     23
      ⋮
 999931
 999953
 999959
 999961
 999969
 999979
 999981
 999983
 999993

In [61]:
function isPrime7(n::Integer)
  if n==1
    return false
  end
  # get all primes up to the square root of n
  for k in getPrimes3(floor(Int,sqrt(n)))
    if n%k==0
      return false
    end
  end
  true
end

isPrime7 (generic function with 1 method)

In [62]:
@btime isPrime7(1_000_003)

  2.759 μs (107 allocations: 12.06 KiB)


true

In [63]:
n = 1_000_000_007

1000000007

In [64]:
isPrime(n)

true

In [65]:
@btime isPrime(n)
@btime isPrime2(n)
@btime isPrime3(n)
@btime isPrime4(n)
@btime isPrime5(n)
@btime isPrime6(n)
@btime isPrime7(n)

  553.519 ms (5 allocations: 7.45 GiB)
  499.142 ms (3 allocations: 128 bytes)
  257.003 ms (3 allocations: 128 bytes)
  15.916 μs (2 allocations: 80 bytes)
  15.625 μs (0 allocations: 0 bytes)
  7.885 μs (0 allocations: 0 bytes)
  75.708 μs (562 allocations: 226.84 KiB)


true

In [96]:
using Primes

In [98]:
@btime isprime(n)

  1.154 μs (0 allocations: 0 bytes)


true

In [49]:
n = nextprime(1_000_000_000)

1000000007

In [60]:
isPrime6(big(2)^89-1)

InterruptException: InterruptException:

In [61]:
@btime isprime(big(2)^89-1)

  9.875 μs (12 allocations: 2.66 KiB)


true